# 04 Score Candidate Pairs
This notebook scores each candidate pair using name similarity, semantic embedding similarity, country match, and city match, then writes the weighted composite scores to `company_er_scores`.

In [0]:
%pip install jellyfish
dbutils.library.restartPython()

In [0]:
# Load candidate pairs and define scoring weights and models.
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import jellyfish

source_table = "workspace.entity_resolution_project.company_er_candidates"
target_table = "workspace.entity_resolution_project.company_er_scores"

EMBEDDING_MODEL = "databricks-gte-large-en"

NAME_WEIGHT = 0.60
SEMANTIC_WEIGHT = 0.05
COUNTRY_WEIGHT = 0.20
CITY_WEIGHT = 0.15

candidates = spark.table(source_table)

In [0]:
# Define the Jaro-Winkler name similarity function.
@F.udf(DoubleType())
def jaro_winkler_similarity(a, b):
    if a is None or b is None:
        return 0.0

    a = str(a).strip()
    b = str(b).strip()

    if not a or not b:
        return 0.0

    return float(jellyfish.jaro_winkler_similarity(a, b))

In [0]:
 # Build embedding text and generate embeddings for input and candidate companies.
left_texts = (
    candidates
    .select(
        "left_row_key",
        "left_company_name",
        "left_country_code",
        "left_country",
        "left_city",
        "left_search_text"
    )
    .dropDuplicates(["left_row_key"])
    .withColumn(
        "left_embedding_text",
        F.concat_ws(
            " | ",
            F.coalesce(F.col("left_company_name"), F.lit("")),
            F.coalesce(F.col("left_country"), F.lit("")),
            F.coalesce(F.col("left_country_code"), F.lit("")),
            F.coalesce(F.col("left_city"), F.lit("")),
            F.coalesce(F.col("left_search_text"), F.lit(""))
        )
    )
    .select("left_row_key", "left_embedding_text")
)

right_texts = (
    candidates
    .select(
        "right_row_key",
        "right_company_name",
        "right_country",
        "right_country_code",
        "right_city",
        "right_website_domain",
        "right_search_text"
    )
    .dropDuplicates(["right_row_key"])
    .withColumn(
        "right_embedding_text",
        F.concat_ws(
            " | ",
            F.coalesce(F.col("right_company_name"), F.lit("")),
            F.coalesce(F.col("right_country"), F.lit("")),
            F.coalesce(F.col("right_country_code"), F.lit("")),
            F.coalesce(F.col("right_city"), F.lit("")),
            F.coalesce(F.col("right_website_domain"), F.lit("")),
            F.coalesce(F.col("right_search_text"), F.lit(""))
        )
    )
    .select("right_row_key", "right_embedding_text")
)

left_embeddings = left_texts.selectExpr(
    "*",
    f"ai_query('{EMBEDDING_MODEL}', left_embedding_text) AS left_embedding"
)

right_embeddings = right_texts.selectExpr(
    "*",
    f"ai_query('{EMBEDDING_MODEL}', right_embedding_text) AS right_embedding"
)

In [0]:
# Calculate name, semantic, country, city, and composite match scores.
candidates_with_embeddings = (
    candidates
    .join(left_embeddings, on="left_row_key", how="left")
    .join(right_embeddings, on="right_row_key", how="left")
)

scores_with_embeddings = (
    candidates_with_embeddings
    .withColumn(
        "name_similarity",
        jaro_winkler_similarity(
            F.col("left_clean_company_name"),
            F.col("right_clean_company_name")
        )
    )
    .withColumn(
        "country_match",
        F.when(
            (F.col("left_country_code").isNotNull()) &
            (F.col("right_country_code").isNotNull()) &
            (F.col("left_country_code") == F.col("right_country_code")),
            F.lit(1.0)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "city_match",
        F.when(
            (F.col("left_city").isNotNull()) &
            (F.col("right_city").isNotNull()) &
            (F.col("left_city") == F.col("right_city")),
            F.lit(1.0)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "semantic_dot",
        F.expr(
            """
            aggregate(
              zip_with(left_embedding, right_embedding, (x, y) -> x * y),
              cast(0.0 as double),
              (acc, z) -> acc + z
            )
            """
        )
    )
    .withColumn(
        "left_norm",
        F.sqrt(
            F.expr(
                """
                aggregate(
                  transform(left_embedding, x -> x * x),
                  cast(0.0 as double),
                  (acc, z) -> acc + z
                )
                """
            )
        )
    )
    .withColumn(
        "right_norm",
        F.sqrt(
            F.expr(
                """
                aggregate(
                  transform(right_embedding, x -> x * x),
                  cast(0.0 as double),
                  (acc, z) -> acc + z
                )
                """
            )
        )
    )
    .withColumn(
        "semantic_similarity_raw",
        F.when(
            (F.col("left_norm") > 0) & (F.col("right_norm") > 0),
            F.col("semantic_dot") / (F.col("left_norm") * F.col("right_norm"))
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "semantic_similarity",
        F.greatest(
            F.lit(0.0),
            F.least(F.lit(1.0), F.col("semantic_similarity_raw"))
        )
    )
    .withColumn(
        "composite_score",
        (
            F.col("name_similarity") * F.lit(NAME_WEIGHT) +
            F.col("semantic_similarity") * F.lit(SEMANTIC_WEIGHT) +
            F.col("country_match") * F.lit(COUNTRY_WEIGHT) +
            F.col("city_match") * F.lit(CITY_WEIGHT)
        )
    )
)

scores = scores_with_embeddings.drop(
    "left_embedding",
    "right_embedding",
    "semantic_dot",
    "left_norm",
    "right_norm",
    "semantic_similarity_raw"
)

In [0]:
# Save scored candidates and review the highest-scoring pairs.
(
    scores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

display(
    scores
    .select(
        "left_row_key",
        "right_row_key",
        "left_company_name",
        "right_company_name",
        "left_country_code",
        "right_country_code",
        "left_city",
        "right_city",
        "name_similarity",
        "semantic_similarity",
        "country_match",
        "city_match",
        "composite_score"
    )
    .orderBy(F.desc("composite_score"))
)

print("Scored candidate rows:", scores.count())
print("Input records:", scores.select("left_row_key").distinct().count())